# Git Repository Code Quality Analysis
Analyses code complexity (cyclomatic) and rate of change per file, displayed as an infographic.

In [ ]:
import subprocess
import os
import tempfile
import datetime
from pathlib import Path
from collections import defaultdict

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
from radon.complexity import cc_visit, cc_rank
from radon.metrics import mi_visit
import git
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────────
# For a public repo:
#   REPO_URL = "https://github.com/org/repo.git"
# For a private repo with a personal access token:
#   REPO_URL = "https://<TOKEN>@github.com/org/repo.git"
# Or point to an already-cloned local directory:
#   REPO_URL = None
#   REPO_LOCAL_PATH = Path("/path/to/local/repo")

REPO_URL        = "git@github.com:REPO/investor-portal-backend.git"
REPO_LOCAL_PATH = '/Users/julianharley/Projects/git-analyis/repository/'   # set to override cloning, e.g. Path("/Users/you/repos/investor-portal-backend")

if REPO_LOCAL_PATH and Path(REPO_LOCAL_PATH).exists():
    REPO_DIR = Path(REPO_LOCAL_PATH)
    print(f"Using local repo: {REPO_DIR}")
elif REPO_URL:
    REPO_DIR = Path(tempfile.mkdtemp()) / "repo"
    print(f"Will clone {REPO_URL} → {REPO_DIR}")
else:
    raise ValueError("Set either REPO_URL or REPO_LOCAL_PATH")

In [ ]:
# Clone the repository (skipped if using local path)
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    repo = git.Repo(REPO_DIR)
    print(f"Using existing repo at {REPO_DIR}")
else:
    repo = git.Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Cloned.")

print(f"Active branch : {repo.active_branch.name}")
print(f"Total commits : {len(list(repo.iter_commits()))}")

In [ ]:
# ── Collect git history per file ──────────────────────────────────────────────
EXCLUDE_DIRS = {
    "migrations",
    "__pycache__",
    ".git",
    "venv", ".venv",
    "node_modules",
}
EXCLUDE_FILES = {"__init__.py"}

def is_excluded(filepath: str) -> bool:
    parts = filepath.split("/")
    return bool(set(parts) & EXCLUDE_DIRS) or parts[-1] in EXCLUDE_FILES


file_commits: dict[str, list] = defaultdict(list)
file_churn:   dict[str, int]  = defaultdict(int)

for commit in repo.iter_commits():
    commit_dt = datetime.datetime.fromtimestamp(commit.committed_date, tz=datetime.timezone.utc)
    try:
        stats = commit.stats.files
    except Exception:
        continue
    for filepath, fstats in stats.items():
        if filepath.endswith(".py") and not is_excluded(filepath):
            file_commits[filepath].append(commit_dt)
            file_churn[filepath] += fstats.get("insertions", 0) + fstats.get("deletions", 0)

print(f"Tracked {len(file_commits)} Python files in git history (migrations + __init__.py excluded)")

In [ ]:
# ── Compute cyclomatic complexity & maintainability index per file ─────────────

def analyse_file(abs_path: Path) -> dict:
    """Return avg cyclomatic complexity, max CC, and maintainability index for a Python file."""
    try:
        source = abs_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return {}

    try:
        blocks = cc_visit(source)
        if blocks:
            complexities = [b.complexity for b in blocks]
            avg_cc = sum(complexities) / len(complexities)
            max_cc = max(complexities)
            rank = cc_rank(max_cc)
        else:
            avg_cc, max_cc, rank = 1.0, 1, "A"
    except Exception:
        avg_cc, max_cc, rank = None, None, None

    try:
        mi = mi_visit(source, multi=True)
    except Exception:
        mi = None

    lines = source.splitlines()
    loc = len([l for l in lines if l.strip() and not l.strip().startswith("#")])

    return {"avg_cc": avg_cc, "max_cc": max_cc, "cc_rank": rank, "mi": mi, "loc": loc}


rows = []
for py_file in REPO_DIR.rglob("*.py"):
    rel = py_file.relative_to(REPO_DIR).as_posix()
    if is_excluded(rel):        # skip migrations and other generated dirs
        continue
    metrics = analyse_file(py_file)
    if not metrics:
        continue

    commits = file_commits.get(rel, [])
    n_commits = len(commits)
    churn = file_churn.get(rel, 0)

    if n_commits >= 2:
        first_seen  = min(commits)
        last_seen   = max(commits)
        age_days    = max((last_seen - first_seen).days, 1)
        change_rate = round(n_commits / age_days * 30, 3)
    elif n_commits == 1:
        first_seen  = last_seen = commits[0]
        age_days    = 0
        change_rate = 0.0
    else:
        first_seen = last_seen = None
        age_days   = 0
        change_rate = 0.0

    rows.append({
        "file":        rel,
        "module":      "/".join(rel.split("/")[:-1]) or "root",
        "loc":         metrics["loc"],
        "avg_cc":      metrics["avg_cc"],
        "max_cc":      metrics["max_cc"],
        "cc_rank":     metrics["cc_rank"],
        "mi":          metrics["mi"],
        "n_commits":   n_commits,
        "churn":       churn,
        "change_rate": change_rate,
        "first_seen":  first_seen,
        "last_seen":   last_seen,
        "age_days":    age_days,
    })

print(f"Analysed {len(rows)} files (migrations excluded)")

In [ ]:
# ── Build Polars DataFrame ─────────────────────────────────────────────────────

df = pl.DataFrame(rows).with_columns([
    pl.col("avg_cc").cast(pl.Float64),
    pl.col("max_cc").cast(pl.Int32),
    pl.col("mi").cast(pl.Float64),
    pl.col("change_rate").cast(pl.Float64),
]).sort("avg_cc", descending=True)

print(df.shape)
df.head(200)

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
df.select(["loc","avg_cc","max_cc","mi","n_commits","churn","change_rate"]).describe()

In [ ]:
# ── Infographic ────────────────────────────────────────────────────────────────
pdf = df.to_pandas()
pdf = pdf.dropna(subset=["avg_cc", "mi", "change_rate"])

REPO_NAME = REPO_DIR.name

RANK_COLOUR = {"A": "#2ecc71", "B": "#82e0aa", "C": "#f9e79f",
               "D": "#f0b27a", "E": "#e59866", "F": "#e74c3c"}
colours = [RANK_COLOUR.get(r, "#aab7b8") for r in pdf["cc_rank"]]

fig = plt.figure(figsize=(22, 24), facecolor="#1a1a2e")
fig.suptitle(
    f"Code Quality  ×  Rate of Change\n{REPO_NAME}",
    fontsize=18, fontweight="bold", color="white", y=0.98
)

def style_ax(ax, title=""):
    ax.set_facecolor("#16213e")
    ax.tick_params(colors="#aaaaaa")
    ax.spines["bottom"].set_color("#444")
    ax.spines["left"].set_color("#444")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:
        ax.set_title(title, color="#e0e0e0", fontsize=11, pad=8)

gs = fig.add_gridspec(4, 2, hspace=0.42, wspace=0.3,
                      left=0.06, right=0.97, top=0.94, bottom=0.04)

# ── 1. Bubble chart: complexity vs change rate (size = LOC) ───────────────────
ax1 = fig.add_subplot(gs[0, :])
style_ax(ax1, "Complexity vs Rate of Change  (bubble size = lines of code)")
sizes = np.clip(pdf["loc"] * 1.5, 20, 800)
ax1.scatter(pdf["change_rate"], pdf["avg_cc"],
            s=sizes, c=colours, alpha=0.75, edgecolors="#ffffff30", linewidths=0.4)
threshold_cc = pdf["avg_cc"].quantile(0.80)
threshold_cr = pdf["change_rate"].quantile(0.70)
label_mask   = (pdf["avg_cc"] >= threshold_cc) & (pdf["change_rate"] >= threshold_cr)
for _, row in pdf[label_mask].iterrows():
    fname = row["file"].split("/")[-1].replace(".py", "")
    ax1.annotate(fname, (row["change_rate"], row["avg_cc"]),
                 xytext=(6, 4), textcoords="offset points",
                 fontsize=7, color="#ffffffbb")
ax1.set_xlabel("Commits per month (change rate)", color="#aaaaaa")
ax1.set_ylabel("Avg cyclomatic complexity", color="#aaaaaa")
ax1.axhline(y=5,  color="#f39c12", lw=0.8, ls="--", alpha=0.6)
ax1.axhline(y=10, color="#e74c3c", lw=0.8, ls="--", alpha=0.6)
rank_patches = [mpatches.Patch(color=c, label=f"Rank {r}") for r, c in RANK_COLOUR.items()]
ax1.legend(handles=rank_patches, loc="upper right", ncol=6,
           fontsize=7.5, labelcolor="white", facecolor="#0f3460",
           framealpha=0.8, edgecolor="#444")

# ── 2. Top 20 files by avg CC ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
style_ax(ax2, "Top 20 files — Avg Cyclomatic Complexity")
top20 = pdf.nlargest(20, "avg_cc")
labels2 = [f.split("/")[-1].replace(".py", "") for f in top20["file"]]
bar_c2 = [RANK_COLOUR.get(r, "#aab7b8") for r in top20["cc_rank"]]
bars2 = ax2.barh(range(len(top20)), top20["avg_cc"], color=bar_c2, edgecolor="#ffffff10")
ax2.set_yticks(range(len(top20))); ax2.set_yticklabels(labels2, fontsize=8, color="#cccccc")
ax2.invert_yaxis(); ax2.set_xlabel("Avg CC", color="#aaaaaa")
for bar, val in zip(bars2, top20["avg_cc"]):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}", va="center", fontsize=7, color="#dddddd")

# ── 3. Top 20 files by change rate ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
style_ax(ax3, "Top 20 files — Commits per Month")
top20cr = pdf.nlargest(20, "change_rate")
labels3 = [f.split("/")[-1].replace(".py", "") for f in top20cr["file"]]
bar_c3 = [RANK_COLOUR.get(r, "#aab7b8") for r in top20cr["cc_rank"]]
bars3 = ax3.barh(range(len(top20cr)), top20cr["change_rate"], color=bar_c3, edgecolor="#ffffff10")
ax3.set_yticks(range(len(top20cr))); ax3.set_yticklabels(labels3, fontsize=8, color="#cccccc")
ax3.invert_yaxis(); ax3.set_xlabel("Commits / month", color="#aaaaaa")
for bar, val in zip(bars3, top20cr["change_rate"]):
    ax3.text(val + 0.02, bar.get_y() + bar.get_height()/2,
             f"{val:.2f}", va="center", fontsize=7, color="#dddddd")

# ── 4. Maintainability index distribution ─────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
style_ax(ax4, "Maintainability Index distribution (higher = better)")
mi_vals = pdf["mi"].dropna()
n, bins, patches = ax4.hist(mi_vals, bins=25, edgecolor="#ffffff20")
for patch, left in zip(patches, bins[:-1]):
    patch.set_facecolor("#2ecc71" if left >= 65 else "#f39c12" if left >= 20 else "#e74c3c")
ax4.axvline(65, color="#2ecc71", lw=1, ls="--", label="Maintainable (≥65)")
ax4.axvline(20, color="#e74c3c", lw=1, ls="--", label="Unmaintainable (<20)")
ax4.set_xlabel("Maintainability Index", color="#aaaaaa")
ax4.set_ylabel("# files", color="#aaaaaa")
ax4.legend(fontsize=8, labelcolor="white", facecolor="#0f3460", framealpha=0.8)

# ── 5. CC Rank distribution (donut) ───────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor("#16213e")
ax5.set_title("CC Rank distribution", color="#e0e0e0", fontsize=11, pad=8)
rank_counts = pdf["cc_rank"].value_counts().sort_index()
valid_ranks = [(r, cnt) for r, cnt in rank_counts.items() if r in RANK_COLOUR]
if valid_ranks:
    rlabels, rcounts = zip(*valid_ranks)
    wedges, texts, autotexts = ax5.pie(
        rcounts, labels=rlabels, colors=[RANK_COLOUR[r] for r in rlabels],
        autopct="%1.0f%%", startangle=90,
        wedgeprops=dict(width=0.55, edgecolor="#1a1a2e"),
        textprops=dict(color="white", fontsize=10)
    )
    for at in autotexts: at.set_fontsize(8)

# ── 6. Churn vs LOC scatter ────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[3, :])
style_ax(ax6, "Total churn (lines added + deleted) vs Lines of Code  — coloured by CC rank")
ax6.scatter(pdf["loc"], pdf["churn"],
            s=40, c=[RANK_COLOUR.get(r, "#aab7b8") for r in pdf["cc_rank"]],
            alpha=0.7, edgecolors="#ffffff20", linewidths=0.3)
churn_q90 = pdf["churn"].quantile(0.90)
for _, row in pdf[pdf["churn"] >= churn_q90].iterrows():
    fname = row["file"].split("/")[-1].replace(".py", "")
    ax6.annotate(fname, (row["loc"], row["churn"]),
                 xytext=(5, 3), textcoords="offset points",
                 fontsize=7, color="#ffffffbb")
ax6.set_xlabel("Lines of code (non-blank, non-comment)", color="#aaaaaa")
ax6.set_ylabel("Total churn", color="#aaaaaa")

out_path = Path("code_quality_infographic.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out_path.resolve()}")

In [ ]:
__# ── Full sortable table ────────────────────────────────────────────────────────
# Risk score = avg_cc * log1p(change_rate)
df_risk = df.with_columns(
    (pl.col("avg_cc") * (pl.col("change_rate") + 1).log()).alias("risk_score")
).sort("risk_score", descending=True)

print("Top 30 riskiest files (high complexity + high change rate):")
df_risk.select(["file", "loc", "avg_cc", "max_cc", "cc_rank", "mi",
                "n_commits", "change_rate", "churn", "risk_score"]).head(30)

In [ ]:
# ── Explainer Infographic ──────────────────────────────────────────────────────
# Visual guide to every metric used in this analysis

BG      = "#1a1a2e"
PANEL   = "#16213e"
ACCENT  = "#e0e0e0"
DIM     = "#888888"
GREEN   = "#2ecc71"
YELLOW  = "#f9e79f"
ORANGE  = "#f39c12"
RED     = "#e74c3c"
BLUE    = "#3498db"

RANK_COLOUR = {"A": "#2ecc71", "B": "#82e0aa", "C": "#f9e79f",
               "D": "#f0b27a", "E": "#e59866", "F": "#e74c3c"}

fig = plt.figure(figsize=(22, 26), facecolor=BG)
fig.suptitle("Metric Reference Guide", fontsize=20, fontweight="bold",
             color="white", y=0.99)

def panel(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=ACCENT, fontsize=11, fontweight="bold", pad=10, loc="left")
    ax.tick_params(colors=DIM)
    for spine in ax.spines.values():
        spine.set_color("#333")

gs = fig.add_gridspec(4, 3, hspace=0.55, wspace=0.35,
                      left=0.05, right=0.97, top=0.96, bottom=0.03)

# ── 1. Cyclomatic Complexity scale (A–F) ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
panel(ax1, "Cyclomatic Complexity (avg_cc / max_cc)")

bands = [
    ("A", "1–5",   "Simple, single path\nEasy to test & understand",   RANK_COLOUR["A"]),
    ("B", "6–10",  "Some branching\nStill manageable",                  RANK_COLOUR["B"]),
    ("C", "11–15", "Notable complexity\nConsider refactoring",          RANK_COLOUR["C"]),
    ("D", "16–20", "High complexity\nHard to test fully",               RANK_COLOUR["D"]),
    ("E", "21–25", "Very high\nTest coverage likely patchy",            RANK_COLOUR["E"]),
    ("F", "26+",   "Untestable in practice\nRefactor urgently",         RANK_COLOUR["F"]),
]
for i, (rank, rng, desc, colour) in enumerate(bands):
    x = i / 6
    w = 1 / 6
    ax1.barh(0, w, left=x, height=0.55, color=colour, edgecolor="#1a1a2e", linewidth=1.5)
    ax1.text(x + w/2, 0.42, rank, ha="center", va="center",
             fontsize=18, fontweight="bold", color="#1a1a2e")
    ax1.text(x + w/2, -0.08, rng, ha="center", va="top", fontsize=9, color=ACCENT)
    for j, line in enumerate(desc.split("\n")):
        ax1.text(x + w/2, -0.28 - j*0.18, line, ha="center", va="top",
                 fontsize=7.5, color=DIM)
ax1.set_xlim(0, 1); ax1.set_ylim(-0.9, 0.8)
ax1.axis("off")
ax1.text(0.5, 0.75,
         "Each function/method gets a CC score = 1 + number of branches (if/for/while/except/and/or).",
         ha="center", va="center", fontsize=8.5, color=DIM, transform=ax1.transAxes)

# ── 2. CC: what counts as a branch ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
panel(ax2, "What raises CC?")
ax2.axis("off")
items = [
    ("+1", "if / elif / else branch"),
    ("+1", "for / while loop"),
    ("+1", "except clause"),
    ("+1", "and / or in condition"),
    ("+1", "ternary  (x if c else y)"),
    ("+1", "assert statement"),
    ("= 1", "base score per function"),
]
for i, (badge, text) in enumerate(items):
    y = 0.92 - i * 0.13
    colour = RED if badge == "+1" else BLUE
    ax2.text(0.08, y, badge, fontsize=9, fontweight="bold", color=colour,
             transform=ax2.transAxes, va="center",
             bbox=dict(boxstyle="round,pad=0.25", facecolor="#0f3460", edgecolor=colour))
    ax2.text(0.28, y, text, fontsize=8.5, color=ACCENT,
             transform=ax2.transAxes, va="center")

# ── 3. Maintainability Index ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
panel(ax3, "Maintainability Index (mi)")
zones = [(0, 20, RED, "Unmaintainable\n< 20"),
         (20, 65, ORANGE, "Moderate\n20 – 64"),
         (65, 100, GREEN, "Maintainable\n≥ 65")]
for lo, hi, colour, label in zones:
    ax3.barh(0, hi - lo, left=lo, height=0.4, color=colour,
             edgecolor="#1a1a2e", linewidth=1)
    ax3.text((lo + hi) / 2, 0, label, ha="center", va="center",
             fontsize=8, color="#1a1a2e", fontweight="bold")
ex = 72
ax3.annotate("", xy=(ex, 0.22), xytext=(ex, 0.55),
             arrowprops=dict(arrowstyle="-|>", color="white", lw=1.5))
ax3.text(ex, 0.62, f"e.g. {ex}", ha="center", fontsize=8, color="white")
ax3.set_xlim(0, 100); ax3.set_ylim(-0.5, 1.0)
ax3.set_xticks([0, 20, 65, 100])
ax3.tick_params(axis="x", colors=DIM, labelsize=8)
ax3.set_yticks([])
ax3.spines["left"].set_visible(False)
ax3.text(0.5, -0.22, "Derived from LOC, CC, and Halstead volume.\nHigher = easier to maintain.",
         ha="center", va="top", fontsize=7.5, color=DIM, transform=ax3.transAxes)

# ── 4. LOC ────────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
panel(ax4, "LOC  (lines of code)")
ax4.axis("off")
examples = [
    ("tiny",   30,  GREEN,       "< 50 — Focused, easy to review"),
    ("small",  120, "#82e0aa",   "50–200 — Typical well-scoped file"),
    ("medium", 350, YELLOW,      "200–500 — Getting complex"),
    ("large",  600, ORANGE,      "500–1000 — Consider splitting"),
    ("huge",  1000, RED,         "1000+ — Strong split signal"),
]
bar_h = 0.12
for i, (label, val, colour, note) in enumerate(examples):
    y = 0.88 - i * 0.18
    width = val / 1100
    ax4.barh(y, width, height=bar_h, color=colour,
             transform=ax4.transAxes, left=0.22, clip_on=False)
    ax4.text(0.20, y + bar_h/2, label, ha="right", va="center",
             fontsize=8, color=DIM, transform=ax4.transAxes)
    ax4.text(0.23 + width, y + bar_h/2, note,
             ha="left", va="center", fontsize=7.5, color=ACCENT,
             transform=ax4.transAxes)
ax4.text(0.5, 0.02, "Blank lines and comment-only lines are excluded.",
         ha="center", fontsize=7.5, color=DIM, transform=ax4.transAxes)

# ── 5. n_commits vs churn ─────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
panel(ax5, "n_commits  vs  churn")
ax5.axis("off")

ax5.text(0.5, 0.95, "n_commits", ha="center", fontsize=10,
         fontweight="bold", color=BLUE, transform=ax5.transAxes)
ax5.text(0.5, 0.86,
         "How many commits touched this file.\nHigh = frequently revisited.",
         ha="center", fontsize=8, color=ACCENT, transform=ax5.transAxes)

# Use ax5.plot with transAxes instead of axhline to draw in axes coordinates
ax5.plot([0.1, 0.9], [0.73, 0.73], color="#444", lw=0.8, transform=ax5.transAxes)

ax5.text(0.5, 0.67, "churn", ha="center", fontsize=10,
         fontweight="bold", color=ORANGE, transform=ax5.transAxes)
ax5.text(0.5, 0.58,
         "Total lines added + deleted across all commits.\n"
         "Measures volume of editing, not just frequency.\n"
         "High churn on a complex file is a risk signal.",
         ha="center", fontsize=8, color=ACCENT, transform=ax5.transAxes)

ax5.plot([0.1, 0.9], [0.44, 0.44], color="#444", lw=0.8, transform=ax5.transAxes)

ax5.text(0.5, 0.37,
         "A file can have low n_commits but high churn\n"
         "(one massive rewrite), or high n_commits\n"
         "but low churn (many small tweaks).",
         ha="center", fontsize=7.5, color=DIM, transform=ax5.transAxes, style="italic")

# ── 6. change_rate timeline ───────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
panel(ax6, "change_rate  (commits per month)")
examples_cr = [
    ("migration\n(excluded)",  [0],                  DIM,    0.0,  "introduced once → 0.0"),
    ("stable util",            [0, 60, 180, 300],    GREEN,  0.3,  "rarely touched → low"),
    ("active service",         [0, 14, 30, 45, 60],  ORANGE, 4.2,  "iterated frequently → high"),
    ("hotfix file",            [0, 2, 3, 5, 7, 9],   RED,    12.0, "rapid churn → very high"),
]
y_positions = [0.78, 0.52, 0.26, 0.00]
for (label, days, colour, rate, note), y in zip(examples_cr, y_positions):
    ax6.annotate("", xy=(310, y), xytext=(0, y),
                 arrowprops=dict(arrowstyle="-", color="#444", lw=1),
                 xycoords=("data", "axes fraction"),
                 textcoords=("data", "axes fraction"))
    for d in days:
        ax6.plot(d, y, "o", color=colour, ms=8,
                 transform=ax6.get_xaxis_transform())
    ax6.text(-5, y, label, ha="right", va="center", fontsize=8,
             color=ACCENT, transform=ax6.get_xaxis_transform())
    ax6.text(315, y, f"{rate:.1f} /mo  — {note}", ha="left", va="center",
             fontsize=8, color=colour, transform=ax6.get_xaxis_transform())
ax6.set_xlim(-5, 310); ax6.set_ylim(-0.15, 0.95)
ax6.set_xticks([0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300])
ax6.set_xticklabels(["day 0","30","60","90","120","150","180","210","240","270","300"],
                    fontsize=7.5, color=DIM)
ax6.set_yticks([])
ax6.spines["left"].set_visible(False)
ax6.text(0.5, -0.10,
         "change_rate = n_commits ÷ file_lifespan_days × 30    "
         "(files with only 1 commit score 0.0 — they have never been re-edited)",
         ha="center", fontsize=8, color=DIM, transform=ax6.transAxes, style="italic")

# ── 7. Risk score ─────────────────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
panel(ax7, "risk_score  (composite)")
ax7.axis("off")
ax7.text(0.5, 0.92, "risk_score  =  avg_cc  ×  log(1 + change_rate)",
         ha="center", fontsize=9, fontweight="bold", color=ACCENT,
         transform=ax7.transAxes,
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#0f3460", edgecolor=BLUE))
ax7.text(0.5, 0.76, "Combines how complex a file is\nwith how often it changes.",
         ha="center", fontsize=8.5, color=ACCENT, transform=ax7.transAxes)
quad_data = [
    (0.18, 0.44, GREEN,  "Low CC\nLow rate\n→ Healthy"),
    (0.72, 0.44, YELLOW, "Low CC\nHigh rate\n→ Watch"),
    (0.18, 0.14, ORANGE, "High CC\nLow rate\n→ Refactor\nwhen touched"),
    (0.72, 0.14, RED,    "High CC\nHigh rate\n→ High risk\nprioritise now"),
]
ax7.plot([0.08, 0.92], [0.30, 0.30], color="#444", lw=0.7, transform=ax7.transAxes)
ax7.plot([0.50, 0.50], [0.05, 0.57], color="#444", lw=0.7, transform=ax7.transAxes)
for x, y, colour, text in quad_data:
    ax7.text(x, y, text, ha="center", va="center", fontsize=7.5,
             color=colour, transform=ax7.transAxes, fontweight="bold")
ax7.text(0.08, 0.60, "CC →", fontsize=7, color=DIM, transform=ax7.transAxes)
ax7.text(0.01, 0.30, "rate\n↑", fontsize=7, color=DIM, transform=ax7.transAxes, ha="center")

# ── 8. Glossary strip ─────────────────────────────────────────────────────────
ax8 = fig.add_subplot(gs[3, :])
panel(ax8, "Quick Reference")
ax8.axis("off")
glossary = [
    ("LOC",         "Non-blank, non-comment lines of source code"),
    ("avg_cc",      "Mean cyclomatic complexity across all functions in the file"),
    ("max_cc",      "Complexity of the single most complex function — worst-case testability"),
    ("cc_rank",     "Radon letter grade for max_cc  (A = simplest … F = most complex)"),
    ("mi",          "Maintainability Index 0–100; combines LOC, CC, Halstead volume"),
    ("n_commits",   "Number of git commits that touched this file"),
    ("churn",       "Sum of lines inserted + deleted across all commits"),
    ("change_rate", "Commits per month over the file's active lifespan (0 if never re-edited)"),
    ("risk_score",  "avg_cc × log(1 + change_rate) — higher means more urgent attention"),
]
for i, (term, defn) in enumerate(glossary):
    col = i % 3
    row = i // 3
    x = 0.01 + col * 0.335
    y = 0.88 - row * 0.28
    ax8.text(x, y, term, fontsize=9, fontweight="bold", color=BLUE,
             transform=ax8.transAxes, va="top")
    ax8.text(x, y - 0.10, defn, fontsize=8, color=ACCENT,
             transform=ax8.transAxes, va="top")

plt.savefig("code_quality_explainer.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: code_quality_explainer.png")

In [ ]:
# ── Logical Coupling Analysis ──────────────────────────────────────────────────
# Files that frequently change together in the same commit are "logically coupled".
# High coupling suggests hidden dependencies, shared responsibility, or SRP violations.
# Method: for every commit, enumerate all pairs of Python files that changed together
# and count co-occurrences.

from itertools import combinations

pair_count:       dict[tuple, int] = defaultdict(int)  # (file_a, file_b) -> co-commits
file_commit_sets: dict[str, set]   = defaultdict(set)  # file -> {commit shas}

for commit in repo.iter_commits():
    try:
        changed = [f for f in commit.stats.files if f.endswith(".py") and not is_excluded(f)]
    except Exception:
        continue
    for f in changed:
        file_commit_sets[f].add(commit.hexsha)
    # Only consider commits that touch more than one relevant file
    if len(changed) < 2:
        continue
    for a, b in combinations(sorted(changed), 2):
        pair_count[(a, b)] += 1

print(f"Found {len(pair_count):,} coupled file pairs across {len(file_commit_sets)} files")

# ── Build coupling DataFrame ───────────────────────────────────────────────────
coupling_rows = []
for (a, b), count in pair_count.items():
    commits_a = len(file_commit_sets.get(a, set()))
    commits_b = len(file_commit_sets.get(b, set()))
    # Coupling strength: co-commits relative to the less-frequently-committed file
    # (similar to Jaccard similarity on commit sets)
    union = len(file_commit_sets.get(a, set()) | file_commit_sets.get(b, set()))
    strength = round(count / union, 3) if union else 0.0
    coupling_rows.append({
        "file_a":        a,
        "file_b":        b,
        "co_commits":    count,
        "commits_a":     commits_a,
        "commits_b":     commits_b,
        "coupling_strength": strength,   # 0–1; higher = more tightly coupled
    })

df_coupling = (
    pl.DataFrame(coupling_rows)
    .sort("co_commits", descending=True)
)

# ── Per-file coupling degree ───────────────────────────────────────────────────
# How many distinct partners does each file co-change with?
degree: dict[str, int] = defaultdict(int)
for (a, b) in pair_count:
    degree[a] += 1
    degree[b] += 1

df_degree = (
    pl.DataFrame([{"file": f, "coupling_degree": d} for f, d in degree.items()])
    .sort("coupling_degree", descending=True)
)

print("\nTop 30 most coupled file pairs (by co-commit count):")
df_coupling.head(30)

In [ ]:
# ── Coupling tables & heatmap ─────────────────────────────────────────────────

print("Top 30 files by coupling degree (most co-change partners):")
display(df_degree.head(30))

# ── Heatmap of the top-N most coupled files ───────────────────────────────────
TOP_N = 25  # files to include in the heatmap

top_files = df_degree.head(TOP_N)["file"].to_list()
labels    = [f.split("/")[-1].replace(".py", "") for f in top_files]

# Build NxN matrix
import numpy as np
n = len(top_files)
matrix = np.zeros((n, n))
idx = {f: i for i, f in enumerate(top_files)}

for (a, b), count in pair_count.items():
    if a in idx and b in idx:
        matrix[idx[a], idx[b]] = count
        matrix[idx[b], idx[a]] = count

# Mask the diagonal (self-coupling is meaningless)
np.fill_diagonal(matrix, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(22, 10), facecolor="#1a1a2e",
                          gridspec_kw={"width_ratios": [2, 1]})

# ── Left: co-commit heatmap ────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor("#16213e")

masked = np.ma.masked_invalid(matrix)
cmap = plt.cm.YlOrRd
cmap.set_bad(color="#1a1a2e")

im = ax.imshow(masked, cmap=cmap, aspect="auto")
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.ax.tick_params(colors="#aaaaaa")
cb.set_label("co-commits", color="#aaaaaa")

ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha="right",
                                              fontsize=7.5, color="#cccccc")
ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=7.5, color="#cccccc")
ax.set_title(f"Logical Coupling Heatmap — top {TOP_N} files by degree",
             color="#e0e0e0", fontsize=12, pad=10)
ax.tick_params(colors="#555")
for spine in ax.spines.values():
    spine.set_color("#333")

# Annotate strong couplings
thresh = np.nanpercentile(matrix, 90)
for i in range(n):
    for j in range(n):
        if not np.isnan(matrix[i, j]) and matrix[i, j] >= thresh:
            ax.text(j, i, int(matrix[i, j]), ha="center", va="center",
                    fontsize=6, color="#1a1a2e", fontweight="bold")

# ── Right: top 20 pairs bar chart ─────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#16213e")

top20p = df_coupling.head(20).to_pandas()
pair_labels = [
    f"{a.split('/')[-1].replace('.py','')}  ↔  {b.split('/')[-1].replace('.py','')}"
    for a, b in zip(top20p["file_a"], top20p["file_b"])
]

# Colour by coupling_strength
norm   = plt.Normalize(top20p["coupling_strength"].min(), top20p["coupling_strength"].max())
colours = plt.cm.YlOrRd(norm(top20p["coupling_strength"].values))

bars = ax2.barh(range(len(top20p)), top20p["co_commits"], color=colours, edgecolor="#ffffff10")
ax2.set_yticks(range(len(top20p)))
ax2.set_yticklabels(pair_labels, fontsize=7.5, color="#cccccc")
ax2.invert_yaxis()
ax2.set_xlabel("co-commits", color="#aaaaaa")
ax2.set_title("Top 20 coupled pairs", color="#e0e0e0", fontsize=12, pad=10)
ax2.tick_params(colors="#aaaaaa")
for spine in ax2.spines.values():
    spine.set_color("#333")

for bar, row in zip(bars, top20p.itertuples()):
    ax2.text(bar.get_width() + 0.3,
             bar.get_y() + bar.get_height() / 2,
             f"{row.co_commits}  ({row.coupling_strength:.2f})",
             va="center", fontsize=7, color="#dddddd")

sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=norm)
sm.set_array([])
cb2 = fig.colorbar(sm, ax=ax2, fraction=0.04, pad=0.02)
cb2.set_label("coupling strength", color="#aaaaaa")
cb2.ax.tick_params(colors="#aaaaaa")

fig.suptitle("Logical Coupling Analysis  —  files that change together stay together",
             fontsize=14, fontweight="bold", color="white", y=1.01)

plt.tight_layout()
plt.savefig("coupling_analysis.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: coupling_analysis.png")